<a href="https://colab.research.google.com/github/sita-aghasoy33/basic-llm-tutorials/blob/main/ProjectWork_waiter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ağıllı ofisiant botu (RAG + Tools)

Bu tapşırıqda `narxana_restoran.md` faylı əsasında restoran üçün **ağıllı ofisiant botu** quracaqsınız. Bot:

1. restoran haqqında adi sualları **RAG** ilə cavablandırır;
2. müştəriyə tanış olmayan inqrediyentin **şəklini generasiya edir**;
3. menuya əsasən yeməklər təklif edir.

> **Əsas qayda:** LLM heç bir hesablamanı "başında" etmir. Hər rəqəm ya menyu cədvəlindən, ya da kalkulyator tool-undan gəlir.

## 0. Hazırlıq

**Colab-da:** `Runtime → Change runtime type → T4 GPU` seçin (şəkil generasiyası üçün lazımdır). Sonra `narxana_restoran.md` faylını sol paneldəki **Files → Upload** ilə yükləyin.

**Model:** Tool calling bütün modellərdə işləmir. [openrouter.ai/models](https://openrouter.ai/models) səhifəsində **"Supported parameters → tools"** filtri ilə uyğun model seçin. Məsələn: `openai/gpt-4o-mini`.

In [ ]:
# install and inport necessary packages

In [ ]:
# config

# determine models for chat and for embedding, assign them to a variable

# add apis if you do not work with hugging face

In [ ]:
# read md file with open function

## Hissə 1. RAG bazası - ümumi hissə üçün

Sənəd `##` (bölmə), `###` (alt bölmə) və `####` (yemək) başlıqlarından ibarətdir. Hər chunk-ın əvvəlinə **başlıq yolunu** (header prefix) əlavə edirik, məsələn:

```
Menyu > Desertlər > Nar sorbeti
**Qiymət:** 5.00 AZN · ...
```

Bu olmasa, `**Tərkibi:** təzə nar şirəsi...` kimi chunk hansı yeməyə aid olduğunu "unudur".

`paraphrase-multilingual-MiniLM-L12-v2` modelinin giriş limiti **128 token**-dir. Buna görə limiti keçən bölmələri sətir-sətir bölürük və hər hissəyə yenə prefix əlavə edirik.

In [ ]:
# split to chunks

# embedding

# indexing

## Hissə 2. Menyu chunklarını digərlərindən ayırmaq.

Menyu üçün ayrı bir retriever funksiyası yazın. Sonra ən son funksiyada if statementlə istifadəçi sorğusunda menu sözü varsa, bizi bu retriver funksiyasına yönləndirsin.

## Hissə 3. İnqrediyent şəkli — `generate_image`

- Model: `stabilityai/sdxl-turbo`, 1 addımda şəkil yaradır və T4 GPU-da ~2 saniyə çəkir.
- Prompt **ingiliscə** olmalıdır. Tərcüməni LLM özü edir: tool-un `prompt_en` parametri.
- Eyni inqrediyent üçün şəkil yenidən generasiya olunmur, nəticə **keşdən** götürülür.

Başqa model də istifadə edə bilərsiniz.

In [ ]:
# Hugging face-lə şəkil generasiyası funksiyası yaradın

## Hissə 4. Final funksiyanı yaratmaq


Sistem promptda dil və format məhdudiyyətləri əlavə edin.

final botun görəcəyi işlər:

- müştərinin suallarına cavab vermək
- menyu ilə bağlı suallar olduqda onlara menyuya istinadən cavab vermək
- müştərinin tanımadığı yeməyin və ya inqridientin şəklini generasiya etmək (promptda generate sözü varsa, if vasitəsilə şəkil generasiyası funksiyasına yönləndirmə etmək)


- history management hissəsini list vasitəsilə edə bilərsiniz, dialoqda şəkil generasiyası zamanı şəkil məlumatını history-yə append etməyin. Onun əvəvzinə "generated image" kimi bir cümlə əlavə edə bilərsiniz.

- history çox şişməsin deyə, listdən əvvəlki dialoqları silərək, elə edin ki, sadəcə son 3 yaxud 5 dialoq cütlüyü (5 user, 5 assistant mesajı və 1 sistem mesajı) yadda qalsın.

- funksiya işə düşərkən print yaxud input vasitəsilə istifadəçiyə menyudan sual verəndə menu, şəkil generasiyası istəyəndə isə mütləq generate açar spzündən istifadə etməsini bildirin. Dialoq formatı üçün, istifadəçi dialoqdan çıxmaq istərsə, exit, yaxud quit kimi hansı sözdən istifadə etməli olduğunu bildirin.


Sonda əlinizdə iki funksiya olsun. Biri bir dəfə sual yazıb test etmək üçün, biri də davamlı conversational dialoq aparmaq üçün. History ancaq dialoq üçün lazım olacaq.

In [ ]:
SYSTEM_PROMPT = """
TODO: Sistem promptunu yazın. Aşağıdakıların hamısını əhatə etməlidir: promptu yazanda bu mətni silərsiniz.
"""

## Hissə 5. Test ssenariləri

10 fərqli sualla RAG botu yoxlayın.